# CSC4093/DSC4213: Neural Networks and Deep Learning
## Programming Assignment 01 - LSTMs
### Personal Health Mention Classification from Tweets
---
**Task:** Classify tweets as Personal Health Mentions (1) or Non-Personal Health Mentions (0)  
**Models:** LSTM and Bidirectional LSTM (Bi-LSTM) with Hyperparameter Tuning

## Step 1: Import Libraries

In [ ]:
# Import all libraries needed
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

nltk.download('stopwords')
english_stops = set(stopwords.words('english'))

print('All libraries imported successfully.')

## Step 2: Load and Preview the Dataset

In [ ]:
# Load training and test datasets
train_data = pd.read_csv('phm_train.csv')
test_data  = pd.read_csv('phm_test.csv')

print('Training set shape:', train_data.shape)
print('Test set shape    :', test_data.shape)
print()
print('Training set preview:')
print(train_data.head())
print()
print('Label distribution in training set:')
print(train_data['label'].value_counts())
print()
print('Label distribution in test set:')
print(test_data['label'].value_counts())

## Step 3: Text Preprocessing

In [ ]:
def preprocess_tweets(data):
    x_data = data['tweet'].copy()
    y_data = data['label'].copy()

    # Remove URLs
    x_data = x_data.replace({'http\S+|www\S+': ''}, regex=True)
    # Remove user_mention placeholders
    x_data = x_data.replace({'user_mention': ''}, regex=True)
    # Remove HTML tags
    x_data = x_data.replace({'<.*?>': ''}, regex=True)
    # Remove non-alphabet characters
    x_data = x_data.replace({'[^A-Za-z]': ' '}, regex=True)
    # Tokenize, remove stopwords, lowercase, remove single-char words
    x_data = x_data.apply(lambda tweet: [
        w.lower() for w in tweet.split()
        if w.lower() not in english_stops and len(w) > 1
    ])

    return x_data, y_data

x_train, y_train = preprocess_tweets(train_data)
x_test,  y_test  = preprocess_tweets(test_data)

print('Tweets (Training):')
print(x_train.head())
print()
print('Labels (Training):')
print(y_train.head())

## Step 4: Tokenization and Padding

In [ ]:
def get_max_length(x_data):
    # Use 90th percentile instead of mean — captures more context per tweet
    tweet_lengths = [len(tweet) for tweet in x_data]
    return int(np.percentile(tweet_lengths, 90))

# Fit tokenizer on training data only
token = Tokenizer(lower=False)
token.fit_on_texts(x_train)

x_train_seq = token.texts_to_sequences(x_train)
x_test_seq  = token.texts_to_sequences(x_test)

max_length  = get_max_length(x_train)
total_words = len(token.word_index) + 1

x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Vocabulary size      :', total_words)
print('Max tweet length     :', max_length)
print('Encoded X Train shape:', x_train_pad.shape)
print('Encoded X Test shape :', x_test_pad.shape)
print()
print('Encoded X Train (first 3 rows):')
print(x_train_pad[:3])

## Step 5: Handle Class Imbalance

In [ ]:
# Compute class weights to handle imbalanced dataset
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weight_dict = dict(enumerate(class_weights))

print('Class Weights:', class_weight_dict)
print('  0 (Non-Personal Health) weight:', round(class_weight_dict[0], 4))
print('  1 (Personal Health)     weight:', round(class_weight_dict[1], 4))

## Step 6: Define Shared Hyperparameters

In [ ]:
# ── Shared Hyperparameters ──────────────────────────────────────────
EMBED_DIM   = 128     # Larger embedding captures richer word representations
LSTM_OUT    = 128     # LSTM units
BILSTM_OUT  = 64      # Bi-LSTM units (smaller to prevent overfitting — doubled internally)
DROPOUT     = 0.4     # Dropout for LSTM
DROPOUT_BI  = 0.5     # Slightly higher dropout for Bi-LSTM
BATCH_SIZE  = 32      # Smaller batch = better generalization
EPOCHS      = 20      # EarlyStopping will prevent over-training
LR_LSTM     = 0.001   # Learning rate for LSTM
LR_BILSTM   = 0.0005  # Lower LR for Bi-LSTM — more stable training
# ────────────────────────────────────────────────────────────────────
print('Hyperparameters set.')

## Step 7: Build Model 1 — LSTM (Tuned)

In [ ]:
import os
os.makedirs('models', exist_ok=True)

# LSTM Architecture — Stacked with tuned hyperparameters
lstm_model = Sequential(name='LSTM_Model')
lstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
lstm_model.add(Dropout(DROPOUT))
lstm_model.add(LSTM(LSTM_OUT, return_sequences=True))   # First LSTM layer
lstm_model.add(Dropout(DROPOUT))
lstm_model.add(LSTM(64, return_sequences=False))         # Second stacked LSTM layer
lstm_model.add(Dropout(DROPOUT))
lstm_model.add(Dense(64, activation='relu'))
lstm_model.add(Dense(1,  activation='sigmoid'))

lstm_model.compile(
    optimizer=Adam(learning_rate=LR_LSTM),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(lstm_model.summary())

## Step 8: Train LSTM Model

In [ ]:
lstm_callbacks = [
    ModelCheckpoint(
        'models/LSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

lstm_history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=lstm_callbacks,
    verbose=1
)

## Step 9: Build Model 2 — Bi-LSTM (Tuned)

In [ ]:
# Bi-LSTM Architecture — Stacked with tuned hyperparameters
bilstm_model = Sequential(name='BiLSTM_Model')
bilstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
bilstm_model.add(Dropout(DROPOUT_BI))
bilstm_model.add(Bidirectional(LSTM(BILSTM_OUT, return_sequences=True)))  # First Bi-LSTM layer
bilstm_model.add(Dropout(DROPOUT_BI))
bilstm_model.add(Bidirectional(LSTM(32, return_sequences=False)))          # Second stacked Bi-LSTM layer
bilstm_model.add(Dropout(DROPOUT_BI))
bilstm_model.add(Dense(64, activation='relu'))
bilstm_model.add(Dense(1,  activation='sigmoid'))

bilstm_model.compile(
    optimizer=Adam(learning_rate=LR_BILSTM),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(bilstm_model.summary())

## Step 10: Train Bi-LSTM Model

In [ ]:
bilstm_callbacks = [
    ModelCheckpoint(
        'models/BiLSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=4,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        verbose=1
    )
]

bilstm_history = bilstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=class_weight_dict,
    callbacks=bilstm_callbacks,
    verbose=1
)

## Step 11: Evaluate Both Models on Test Set

In [ ]:
# --- LSTM Evaluation ---
lstm_pred_probs = lstm_model.predict(x_test_pad)
lstm_pred       = (lstm_pred_probs >= 0.5).astype(int)

lstm_correct = sum(1 for i, y in enumerate(y_test) if y == lstm_pred[i])
lstm_acc     = lstm_correct / len(lstm_pred) * 100

print('=== LSTM Model Test Results ===')
print(f'Correct Predictions : {lstm_correct}')
print(f'Wrong Predictions   : {len(lstm_pred) - lstm_correct}')
print(f'Test Accuracy       : {lstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, lstm_pred, target_names=['Non-Personal (0)', 'Personal Health (1)']))

In [ ]:
# --- Bi-LSTM Evaluation ---
bilstm_pred_probs = bilstm_model.predict(x_test_pad)
bilstm_pred       = (bilstm_pred_probs >= 0.5).astype(int)

bilstm_correct = sum(1 for i, y in enumerate(y_test) if y == bilstm_pred[i])
bilstm_acc     = bilstm_correct / len(bilstm_pred) * 100

print('=== Bi-LSTM Model Test Results ===')
print(f'Correct Predictions : {bilstm_correct}')
print(f'Wrong Predictions   : {len(bilstm_pred) - bilstm_correct}')
print(f'Test Accuracy       : {bilstm_acc:.2f}%')
print()
print('Classification Report:')
print(classification_report(y_test, bilstm_pred, target_names=['Non-Personal (0)', 'Personal Health (1)']))

## Step 12: Performance Comparison Table

In [ ]:
lstm_train_acc   = max(lstm_history.history['accuracy'])    * 100
lstm_val_acc     = max(lstm_history.history['val_accuracy']) * 100
lstm_train_loss  = min(lstm_history.history['loss'])
lstm_val_loss    = min(lstm_history.history['val_loss'])

bilstm_train_acc  = max(bilstm_history.history['accuracy'])    * 100
bilstm_val_acc    = max(bilstm_history.history['val_accuracy']) * 100
bilstm_train_loss = min(bilstm_history.history['loss'])
bilstm_val_loss   = min(bilstm_history.history['val_loss'])

comparison = pd.DataFrame({
    'Metric'              : ['Best Training Accuracy (%)', 'Best Validation Accuracy (%)',
                             'Test Accuracy (%)', 'Best Training Loss', 'Best Validation Loss'],
    'LSTM'                : [f'{lstm_train_acc:.2f}',   f'{lstm_val_acc:.2f}',
                             f'{lstm_acc:.2f}',   f'{lstm_train_loss:.4f}',   f'{lstm_val_loss:.4f}'],
    'Bi-LSTM'             : [f'{bilstm_train_acc:.2f}', f'{bilstm_val_acc:.2f}',
                             f'{bilstm_acc:.2f}', f'{bilstm_train_loss:.4f}', f'{bilstm_val_loss:.4f}']
})

print('=== Model Performance Comparison ===')
print(comparison.to_string(index=False))

## Step 13: Accuracy and Loss Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs Bi-LSTM: Training Performance (Tuned)', fontsize=15, fontweight='bold')

# LSTM Accuracy
axes[0, 0].plot(lstm_history.history['accuracy'],     label='Train Accuracy', color='royalblue',  linewidth=2)
axes[0, 0].plot(lstm_history.history['val_accuracy'], label='Val Accuracy',   color='darkorange', linewidth=2)
axes[0, 0].set_title('LSTM - Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# LSTM Loss
axes[0, 1].plot(lstm_history.history['loss'],     label='Train Loss', color='royalblue',  linewidth=2)
axes[0, 1].plot(lstm_history.history['val_loss'], label='Val Loss',   color='darkorange', linewidth=2)
axes[0, 1].set_title('LSTM - Loss')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Loss')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Bi-LSTM Accuracy
axes[1, 0].plot(bilstm_history.history['accuracy'],     label='Train Accuracy', color='seagreen', linewidth=2)
axes[1, 0].plot(bilstm_history.history['val_accuracy'], label='Val Accuracy',   color='crimson',  linewidth=2)
axes[1, 0].set_title('Bi-LSTM - Accuracy')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Accuracy')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Bi-LSTM Loss
axes[1, 1].plot(bilstm_history.history['loss'],     label='Train Loss', color='seagreen', linewidth=2)
axes[1, 1].plot(bilstm_history.history['val_loss'], label='Val Loss',   color='crimson',  linewidth=2)
axes[1, 1].set_title('Bi-LSTM - Loss')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('lstm_bilstm_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved as lstm_bilstm_performance.png')

## Step 14: Discussion

### Model Architectures

#### LSTM (Tuned)
- **Stacked LSTM:** Two LSTM layers (128 → 64 units) allow hierarchical feature extraction from tweet sequences
- **Embedding Dim:** 128 — richer word representation compared to baseline 64
- **Dropout:** 0.4 after each layer to regularize and prevent overfitting on noisy Twitter data
- **Optimizer:** Adam (lr=0.001) with `ReduceLROnPlateau` — learning rate halves when val_loss plateaus
- **Class Weights:** Applied to handle imbalance between personal and non-personal mentions

#### Bi-LSTM (Tuned)
- **Stacked Bi-LSTM:** Two Bidirectional layers (64 → 32 units) — smaller units prevent overfitting since Bi-LSTM doubles parameters internally
- **Higher Dropout:** 0.5 — necessary due to larger effective parameter count from bidirectional processing
- **Lower LR:** 0.0005 — Bi-LSTM benefits from slower, more stable convergence
- **`ReduceLROnPlateau`:** Automatically decays LR when validation loss stagnates

### Key Tuning Changes from Baseline
| Technique | Benefit |
|---|---|
| Stacked LSTM/Bi-LSTM layers | Hierarchical sequence feature learning |
| Embedding dim 64 → 128 | Richer semantic word representations |
| Batch size 64 → 32 | Better gradient estimation, improved generalization |
| `ReduceLROnPlateau` | Prevents training from getting stuck at local minima |
| Class weight balancing | Fairer learning across both health mention classes |
| 90th percentile max length | Captures more tweet context vs. mean-based truncation |
| EarlyStopping patience=4 | Allows more epochs before stopping, avoids premature halt |

### Performance Discussion
The Bi-LSTM, with its ability to process tweet sequences in both forward and backward directions, captures richer contextual signals — particularly useful when health-related keywords appear at the end of a tweet and influence earlier tokens. With the tuned smaller unit size (64 vs 128) and higher dropout (0.5), it is less prone to overfitting compared to the baseline configuration. The standard LSTM, while simpler, benefits greatly from stacking two layers and the larger embedding dimension, enabling it to learn more complex sequential patterns in the informal tweet language.